# Notebook \#2:</br> Exploratory Data Analysis
#### by Sebastian Einar Salas Røkholt

---

**Notebook Index**  

- [**1 - Introduction and Notebook Setup**](#1---introduction-and-notebook-setup)
  - [*1.1 Setup*](#11-setup)
  - [*1.2 Load cleaned dataset*](#12-load-cleaned-dataset)
  - [*1.3 Explanation of variables in the dataset*](#13-explanation-of-variables-in-the-dataset)
- [**2 - Summary Statistics**](#2---summary-statistics)
- [**3 - Correlation Metrics**](#3---correlation-metrics)

## 1 - Introduction and Notebook Setup

### 1.1 Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Notebook settings
%matplotlib inline
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.options.display.float_format = "{:.2f}".format

### 1.2 Load cleaned dataset

In [2]:
df = pd.read_parquet("../Data/etron55-charging-sessions-public-processed.parquet")
df.head()

,charging_id,minutes_elapsed,progress,power,rel_power,d_power,d_power_ema3,soc,d_soc,d_soc_ema3,energy,nominal_power,charger_category,charger_cat_low,charger_cat_mid,charger_cat_high,temp,in_taper,dist_to_taper
0,0,0,0.00,89.44,0.50,0.00,0.00,40.00,0.00,0.00,0.32,150.00,Ultra,0.00,1.00,0.00,4.00,0.00,0.00
1,0,1,0.14,92.75,0.52,3.31,1.66,41.00,1.00,0.50,1.84,150.00,Ultra,0.00,1.00,0.00,4.00,0.00,0.00
2,0,2,0.23,94.81,0.53,2.06,1.86,43.00,2.00,1.25,3.41,150.00,Ultra,0.00,1.00,0.00,4.00,0.00,0.00
3,0,3,0.29,95.68,0.53,0.87,1.36,45.00,2.00,1.62,5.00,150.00,Ultra,0.00,1.00,0.00,4.00,0.00,0.00
4,0,4,0.34,96.88,0.54,1.20,1.28,47.00,2.00,1.81,6.60,150.00,Ultra,0.00,1.00,0.00,4.00,0.00,0.00


### 1.3 Explanation of variables in the dataset
The processed public dataset contains anonymised EV charging sessions with non-sensitive engineered features.

---
 - **`charging_id`, categorical, static (per session):** Identifier for one charging session.
 - **`minutes_elapsed`, numerical, monotonic, time-dependent:** Minutes since session start.
 - **`progress`** — *float ∈ [0,1], time-dependent*: Log-scaled timeline of `minutes_elapsed` capped at 120 min.
---
 - **`power`, numerical, piecewise continuous, time-dependent:** Charging power in kW.
 - **`rel_power`** — *unitless ∈ [0,1], time-dependent*: `power` normalised by `nominal_power`.
 - **`d_power`** — *kW/min, time-dependent*: First difference of `power`.
 - **`d_power_ema3`** — *kW/min, time-dependent*: EMA (span=3) of `d_power`.
---
 - **`soc`, numerical, piecewise continuous, time-dependent:** Battery state of charge in percent.
 - **`d_soc`** — *pp/min, time-dependent*: First difference of `soc`.
 - **`d_soc_ema3`** — *pp/min, time-dependent*: EMA (span=3) of `d_soc`.
---
 - **`energy`, numerical, piecewise continuous, time-dependent:** Delivered session energy in kWh.
---
 - **`nominal_power`** — *kW, ordinal, static*: Charger nameplate capacity.
 - **`charger_category`** — *categorical, static*: Provider category label.
 - **`charger_cat_low`**, **`charger_cat_mid`**, **`charger_cat_high`** — *binary dummies, static*: One-hot bins from `nominal_power`.
---
 - **`temp`, numerical, discrete, static (per session):** Session-level ambient temperature estimate.
 - **`in_taper`**, **`dist_to_taper`** — taper-regime indicators derived from SOC.
---

## 2 - Summary Statistics

In [3]:
# Calculates basic descriptive metrics
print("Shape:", df.shape)
print("Number of charging sessions:", df["charging_id"].nunique())

session_lengths = (df.groupby("charging_id", observed=False)["minutes_elapsed"].max() + 1)
print("Session length summary (minutes):")
print(session_lengths.describe())

print("Temperature summary (°C):")
print(df.groupby("charging_id", observed=False)["temp"].first().describe())

print("\nPandas .info():")
display(df.info())
print("\nPandas .describe():")
display(df.describe())

Shape: (1519679, 19)
Number of charging sessions: 60915
Session length summary (minutes):
count   60915.00
mean       25.20
std        10.57
min         8.00
25%        17.00
50%        23.00
75%        32.00
max        61.00
Name: minutes_elapsed, dtype: float64
Temperature summary (°C):
count   60915.00
mean        7.88
std         8.99
min       -31.00
25%         2.00
50%         8.00
75%        15.00
max        31.00
Name: temp, dtype: float64

Pandas .info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1519679 entries, 0 to 1519678
Data columns (total 19 columns):
 #   Column            Non-Null Count    Dtype   
---  ------            --------------    -----   
 0   charging_id       1519679 non-null  int64   
 1   minutes_elapsed   1519679 non-null  int64   
 2   progress          1519679 non-null  float64 
 3   power             1519679 non-null  float64 
 4   rel_power         1519679 non-null  float64 
 5   d_power           1519679 non-null  float64 
 6   d_power_ema3

None


Pandas .describe():


,charging_id,minutes_elapsed,progress,power,rel_power,d_power,d_power_ema3,soc,d_soc,d_soc_ema3,energy,nominal_power,charger_cat_low,charger_cat_mid,charger_cat_high,temp,in_taper,dist_to_taper
count,1519679.00,1519679.00,1519679.00,1519679.00,1519679.00,1519679.00,1519679.00,1519679.00,1519679.00,1519679.00,1519679.00,1519679.00,1519679.00,1519679.00,1519679.00,1519679.00,1519679.00,1519679.00
mean,6250250.52,14.46,0.51,89.00,0.48,-0.28,-0.19,60.73,1.70,1.64,21.89,175.85,0.13,0.78,0.09,7.56,0.37,0.16
std,3517055.97,10.72,0.18,35.75,0.22,7.23,4.67,21.94,0.95,0.77,16.42,73.42,0.33,0.41,0.29,9.14,0.48,0.27
min,0.00,0.00,0.00,0.00,0.00,-152.18,-75.77,1.00,-1.00,-0.21,-0.01,50.00,0.00,0.00,0.00,-31.00,0.00,0.00
25%,3343043.00,6.00,0.41,59.03,0.31,-2.14,-1.92,44.00,1.00,1.02,8.44,150.00,0.00,1.00,0.00,2.00,0.00,0.00
50%,5958125.00,12.00,0.53,87.21,0.44,0.00,0.00,62.00,2.00,1.64,18.53,180.00,0.00,1.00,0.00,8.00,0.00,0.00
75%,9482876.00,21.00,0.64,119.75,0.63,1.01,1.25,78.00,2.00,2.23,32.23,200.00,0.00,1.00,0.00,14.00,1.00,0.24
max,12657311.00,60.00,0.86,267.79,1.00,160.50,80.25,100.00,9.00,6.00,122.55,500.00,1.00,1.00,1.00,31.00,1.00,1.00


In [4]:
print(df["charger_category"].unique())
print(df["nominal_power"].unique())
print(len(df["nominal_power"].unique()))

['Ultra', 'Rapid']
Categories (2, object): ['Rapid', 'Ultra']
[150. 100.  50.  75.  90. 350.  62. 200. 180.  80. 360. 170. 140. 120.
 110. 240. 280. 400. 500. 300. 275. 115.]
22


## 3 - Correlation Metrics

### 3.1 Correlation of persistance model per horizon

In [5]:
MAX_H = 15  # Maximum prediction horizon
df = df.sort_values(["charging_id","minutes_elapsed"]).copy() 

def persistence_per_horizon(df: pd.DataFrame, max_H: int=1):
    H = range(1, max_H)  # Prediction horizons from 1 to H minutes
    rows = []
    for h in H:
        # Future targets
        yP = df.groupby("charging_id")["power"].shift(-h)
        yS = df.groupby("charging_id")["soc"].shift(-h)
        # Persistence baseline: ŷ_(t+h) = y_(t)
        rmse_pwr = np.sqrt(((yP - df["power"])**2).dropna().mean())
        rmse_soc = np.sqrt(((yS - df["soc"])**2).dropna().mean())
        rows.append({"h":h, "RMSE_persist_power":rmse_pwr, "RMSE_persist_soc":rmse_soc})
    result = pd.DataFrame(rows)
    return result.round(4)


# Calculate persistence baseline RMSE
persistance_baseline = persistence_per_horizon(df, MAX_H)
print("\n\nPersistence baseline RMSE:\n", persistance_baseline)




Persistence baseline RMSE:
      h  RMSE_persist_power  RMSE_persist_soc
0    1                7.39              1.99
1    2               10.12              3.85
2    3               12.80              5.73
3    4               15.40              7.63
4    5               17.94              9.52
5    6               20.41             11.40
6    7               22.78             13.28
7    8               25.07             15.14
8    9               27.28             16.98
9   10               29.41             18.79
10  11               31.46             20.58
11  12               33.42             22.34
12  13               35.28             24.07
13  14               36.98             25.77


### 3.2 Autocorrelation per horizon

In [6]:
def autocorr_per_horizon(df: pd.DataFrame, max_H: int=1):
    H = range(1, max_H)  # Autocorr from 1 to H minutes
    rows = []
    for h in H:
        # Autocorr at lag h
        autocorr_pwr = df.groupby("charging_id")["power"].apply(lambda s: s.autocorr(lag=h)).mean()
        autocorr_soc = df.groupby("charging_id")["soc"].apply(lambda s: s.autocorr(lag=h)).mean()
        rows.append({"h":h, "ACF_power(h)":autocorr_pwr, "ACF_soc(h)":autocorr_soc})
    result = pd.DataFrame(rows)
    return result.round(4)

with np.errstate(divide="ignore", invalid="ignore"):  # Ignore warnings for division by zero due to constant series
    autocorr = autocorr_per_horizon(df, MAX_H)
    print("\n\nAutocorrelation per horizon:\n", autocorr)


/home/srokholt/Masters_Project_Linux_Env/Machine-Teaching-for-XAI--TimeSeries-Models/linux_venv/lib64/python3.12/site-packages/numpy/lib/_function_base_impl.py:2991: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)




Autocorrelation per horizon:
      h  ACF_power(h)  ACF_soc(h)
0    1          0.83        1.00
1    2          0.74        1.00
2    3          0.65        1.00
3    4          0.57        1.00
4    5          0.47        0.99
5    6          0.39        0.99
6    7          0.32        0.99
7    8          0.25        0.99
8    9          0.20        0.99
9   10          0.13        0.99
10  11          0.07        0.99
11  12          0.02        0.99
12  13         -0.04        0.99
13  14         -0.10        0.99


In [7]:
from sklearn.feature_selection import mutual_info_regression

features = ["minutes_elapsed","soc","progress","temp","nominal_power","rel_power",
            "d_power","d_soc","d_power_ema3","d_soc_ema3",
            "charger_cat_low","charger_cat_mid","charger_cat_high"]

def horizon_table(target_col):
    out = []
    for h in range(1,6):
        y = df.groupby("charging_id")[target_col].shift(-h)
        X = df[features].copy()
        mask = y.notna()
        y_ = y[mask].values
        X_ = X[mask].fillna(0).values
        # Pearson/Spearman (monotone)
        corr = pd.Series({f: df.loc[mask, f].corr(y) for f in features}, name=f"corr@h{h}")
        # Mutual Information (nonlinear)
        mi = pd.Series(mutual_info_regression(X_, y_), index=features, name=f"MI@h{h}")
        out.append(pd.concat([corr, mi], axis=1))
    return pd.concat(out, axis=1)

rel_power_tbl = horizon_table("power")
rel_soc_tbl = horizon_table("soc")

# Top drivers per horizon (example for power)
for h in range(1,6):
    display(rel_power_tbl[[f"MI@h{h}"]].sort_values(f"MI@h{h}", ascending=False).head(8))


,MI@h1
rel_power,2.34
d_soc_ema3,1.05
d_power,0.76
soc,0.58
d_power_ema3,0.56
nominal_power,0.47
d_soc,0.36
charger_cat_low,0.21


,MI@h2
rel_power,1.85
d_soc_ema3,0.97
d_power,0.73
soc,0.61
d_power_ema3,0.55
nominal_power,0.47
d_soc,0.35
charger_cat_low,0.21


,MI@h3
rel_power,1.59
d_soc_ema3,0.90
d_power,0.70
soc,0.65
d_power_ema3,0.54
nominal_power,0.48
d_soc,0.34
charger_cat_low,0.21


,MI@h4
rel_power,1.40
d_soc_ema3,0.83
d_power,0.69
soc,0.68
d_power_ema3,0.54
nominal_power,0.48
d_soc,0.32
charger_cat_low,0.21


,MI@h5
rel_power,1.26
d_soc_ema3,0.77
soc,0.71
d_power,0.68
d_power_ema3,0.53
nominal_power,0.48
d_soc,0.30
charger_cat_low,0.21


In [8]:

def taper_soc_of_session(s, drop_ratio=0.95):
    p = s["power"].to_numpy()
    soc = s["soc"].to_numpy()
    if len(p) < 5: return np.nan
    runmax = np.maximum.accumulate(p)
    mask = p < drop_ratio * runmax  # first time power falls 5% below running max
    return soc[np.argmax(mask)] if mask.any() else np.nan

taper = (df.sort_values(["charging_id","minutes_elapsed"])
           .groupby("charging_id", group_keys=False)
           .apply(taper_soc_of_session)
           .rename("taper_soc")
           .to_frame())

print(taper.describe())

# Stratify by temperature quartile to see shifts in taper onset
temp_q = pd.qcut(df.groupby("charging_id")["temp"].median(), 4, labels=False)
taper["temp_q"] = temp_q
print(taper.groupby("temp_q")["taper_soc"].median())


       taper_soc
count   43163.00
mean       69.34
std        13.61
min         4.00
25%        68.00
50%        71.00
75%        79.00
max       100.00
temp_q
0   71.00
1   71.00
2   71.00
3   71.00
Name: taper_soc, dtype: float64


/tmp/ipykernel_3839/1011190228.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(taper_soc_of_session)


In [9]:
# Conditional std of future power given SOC bin at time t, across horizons
std_list = []
for h in range(1,6):
    y = df.groupby("charging_id")["power"].shift(-h)
    tmp = pd.DataFrame({"soc": df["soc"], "y": y}).dropna()
    tmp["soc_bin"] = pd.cut(tmp["soc"], bins=20)
    std = tmp.groupby("soc_bin")["y"].std()
    std.name = f"h{h}"
    std_list.append(std)

cond_std = pd.concat(std_list, axis=1)
print(cond_std.head())  # heatmap in your plotting stack

# Optional: quantile width (P90 - P10) for robustness
qw_list = []
for h in range(1,6):
    y = df.groupby("charging_id")["power"].shift(-h)
    tmp = pd.DataFrame({"soc": df["soc"], "y": y}).dropna()
    tmp["soc_bin"] = pd.cut(tmp["soc"], bins=20)
    q = tmp.groupby("soc_bin")["y"].quantile([.1,.9]).unstack()
    q["width"] = q[0.9] - q[0.1]
    qw_list.append(q["width"].rename(f"h{h}"))
cond_qwidth = pd.concat(qw_list, axis=1)
print(cond_qwidth.head())


/tmp/ipykernel_3839/1686149115.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  std = tmp.groupby("soc_bin")["y"].std()
/tmp/ipykernel_3839/1686149115.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  std = tmp.groupby("soc_bin")["y"].std()
/tmp/ipykernel_3839/1686149115.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  std = tmp.groupby("soc_bin")["y"].std()
/tmp/ipykernel_3839/1686149115.py:7: FutureWarning: The

                 h1    h2    h3    h4  h5
soc_bin                                  
(0.901, 5.95] 32.37 33.12 33.67 34.24 NaN
(5.95, 10.9]  32.55 33.15 33.69 34.16 NaN
(10.9, 15.85] 32.94 33.44 33.89 34.36 NaN
(15.85, 20.8] 33.11 33.60 34.05 34.44 NaN
(20.8, 25.75] 33.31 33.82 34.23 34.57 NaN


/tmp/ipykernel_3839/1686149115.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  q = tmp.groupby("soc_bin")["y"].quantile([.1,.9]).unstack()
/tmp/ipykernel_3839/1686149115.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  q = tmp.groupby("soc_bin")["y"].quantile([.1,.9]).unstack()
/tmp/ipykernel_3839/1686149115.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  q = tmp.groupby("soc_bin")["y"].quantile([.1,.9]).un

                 h1    h2    h3    h4  h5
soc_bin                                  
(0.901, 5.95] 84.89 85.46 86.12 87.64 NaN
(5.95, 10.9]  86.01 88.06 89.85 91.90 NaN
(10.9, 15.85] 87.87 90.75 91.96 92.65 NaN
(15.85, 20.8] 90.40 91.93 92.69 93.27 NaN
(20.8, 25.75] 91.83 92.79 93.51 94.05 NaN


In [10]:
# Label availability per horizon
cov = []
for h in range(1,6):
    yP = df.groupby("charging_id")["power"].shift(-h)
    yS = df.groupby("charging_id")["soc"].shift(-h)
    cov.append({"h":h,
                "N_power_labels": int(yP.notna().sum()),
                "N_soc_labels":   int(yS.notna().sum())})
coverage = pd.DataFrame(cov)
print(coverage)

# Session length distribution (helps choose min length, padding strategy)
lengths = (df.groupby("charging_id")["minutes_elapsed"].max() + 1).rename("length")
print(lengths.describe())


   h  N_power_labels  N_soc_labels
0  1         1458764       1458764
1  2         1397849       1397849
2  3         1336934       1336934
3  4         1276019       1276019
4  5         1215104       1215104
count   60915.00
mean       25.20
std        10.57
min         8.00
25%        17.00
50%        23.00
75%        32.00
max        61.00
Name: length, dtype: float64
